In [1]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")


train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

KNN

In [2]:
from sklearn.metrics import f1_score, classification_report
import torch
import numpy as np
import time
import pandas as pd

X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

class TorchKNNClassifier:
    def __init__(self, n_neighbors=5, val_batch_size=256, train_batch_size=50000, device='cuda'):
        self.k = n_neighbors
        self.val_batch_size = val_batch_size
        self.train_batch_size = train_batch_size
        self.device = device
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        # Chuyển dữ liệu lên GPU một lần để tính toán
        if isinstance(X, pd.DataFrame): X = X.values
        if isinstance(y, pd.Series): y = y.values
        self.X_train = torch.tensor(X, dtype=torch.float32, device=self.device)
        self.y_train = torch.tensor(y, dtype=torch.long, device=self.device)

    def predict(self, X):
        if isinstance(X, pd.DataFrame): X = X.values
        X_val = torch.tensor(X, dtype=torch.float32, device=self.device)
        preds = []
        
        # Chia batch dữ liệu CẦN dự đoán 
        for i in range(0, len(X_val), self.val_batch_size):
            X_batch = X_val[i:i+self.val_batch_size]
            
            # Khởi tạo buffer chứa min distances. Mặc định là vô cực (inf)
            current_top_dists = torch.full((len(X_batch), self.k), float('inf'), device=self.device)
            current_top_indices = torch.zeros((len(X_batch), self.k), dtype=torch.long, device=self.device)
            
            # Phân tách X_train thành các khối nhỏ để lặp qua (Tránh tạo ma trận khổng lồ gây OOM)
            for j in range(0, len(self.X_train), self.train_batch_size):
                X_tr_batch = self.X_train[j:j+self.train_batch_size]
                
                # Tính khoảng cách Euclidean
                dists = torch.cdist(X_batch, X_tr_batch)
                
                # Mức độ cắt k
                k_chunk = min(self.k, dists.shape[1])
                chunk_top_dists, chunk_top_idx = torch.topk(dists, k_chunk, dim=1, largest=False)
                
                # Sửa index thành chỉ số toàn cục trong self.X_train
                chunk_top_idx += j
                
                # Gộp với k nhỏ nhất của các khối trước, sau đó lại top-k một lần nữa
                merged_dists = torch.cat([current_top_dists, chunk_top_dists], dim=1)
                merged_idx = torch.cat([current_top_indices, chunk_top_idx], dim=1)
                
                current_top_dists, best_k_pos = torch.topk(merged_dists, self.k, dim=1, largest=False)
                current_top_indices = torch.gather(merged_idx, 1, best_k_pos)

                # Dọn rác VRAM
                del dists, merged_dists, merged_idx
            
            # Lấy nhãn của k láng giềng gần nhất chung cuộc
            topk_labels = self.y_train[current_top_indices]
            
            # Dự đoán theo nhãn chiếm đa số (mode)
            mode_labels, _ = torch.mode(topk_labels, dim=1)
            preds.append(mode_labels.cpu().numpy())
            
        return np.concatenate(preds)

# Thử các giá trị k khác nhau
k_values = [7]

best_k = None
best_f1 = -1
best_model = None

print("Đang huấn luyện phân lớp KNN (Multi-batch Memory Efficient qua PyTorch) và tìm 'k' tốt nhất...")
start_time = time.time()

for k in k_values:
    # Sử dụng TorchKNNClassifier kết hợp cắt nhỏ ma trận, GPU sẽ an toàn
    knn = TorchKNNClassifier(n_neighbors=k, val_batch_size=256, train_batch_size=50000, device='cuda')
    knn.fit(X_train, y_train)
    
    y_valid_pred = knn.predict(X_valid)
    
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"k = {k:2d} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_k = k
        best_model = knn

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> K tốt nhất tìm được là k = {best_k} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test
print("\n--- Đánh giá Model Tốt nhất (k={}) trên tập TEST ---".format(best_k))
y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp KNN (Multi-batch Memory Efficient qua PyTorch) và tìm 'k' tốt nhất...
k =  7 | Validation Macro F1: 0.5163

=> Quá trình huấn luyện và tuning hoàn tất trong 1.58 giây.
=> K tốt nhất tìm được là k = 7 với Validation Macro F1: 0.5163

--- Đánh giá Model Tốt nhất (k=7) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9195    0.4227    0.5792     20520
           1     0.0588    0.0036    0.0067       281
           2     0.3702    0.3001    0.3315      2709
           3     0.3299    0.2172    0.2619      2763
           4     0.4707    0.7818    0.5876      1132
           5     0.6529    0.2752    0.3872      1210
           6     0.2774    0.9802    0.4324      5048
           7     0.9000    0.8357    0.8667       420

    accuracy                         0.4872     34083
   macro avg     0.4974    0.4771    0.4317     34083
weighted avg     0.7013    0.4872    0.5043     34083



SVM

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...")
start_time = time.time()

# Thử các giá trị alpha (L2 regularization)
alpha_values = [0.0001]

best_alpha = None
best_f1 = -1
best_svm_model = None

# Khối 1: Chuyển dữ liệu lên Numpy 
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đẩy sang Tensor CUDA
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
# Đối với Multi-class SVM, giữ nguyên nhãn số nguyên (0, 1, 2... 10)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Kiến trúc Linear SVM (Multi-class) bằng Pytorch
class TorchLinearSVM(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes) # Sửa output thành 11 classes
        
    def forward(self, x):
        return self.linear(x)

# PyTorch hỗ trợ sẵn MultiMarginLoss dùng để tối ưu Multi-class SVM
criterion = nn.MultiMarginLoss()

for alpha in alpha_values:
    model = TorchLinearSVM(input_dim, num_classes).to(device)
    
    # Khai báo Optimizer - sử dụng weight_decay thay thế L2 penalty
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=alpha)
    
    model.train()
    epochs = 100
    batch_size = 8192
    num_samples = X_train_t.shape[0]
    
    for epoch in range(epochs):
        indices = torch.randperm(num_samples, device=device)
        for i in range(0, num_samples, batch_size):
            idx = indices[i:i+batch_size]
            batch_X = X_train_t[idx]
            batch_y = y_train_t[idx]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            # Tính toán trên GPU với Multi-Class Hinge Loss
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
    
    # Validation step
    model.eval()
    with torch.no_grad():
        valid_outputs = model(X_valid_t)
        # Sử dụng argmax để lấy chỉ số có logit cao nhất (tương ứng với nhãn)
        y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
        
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_alpha = alpha
        best_svm_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên test
print("\n--- Đánh giá Model Multi-class SVM Tốt nhất (alpha={}) trên tập TEST ---".format(best_alpha))
best_svm_model.eval()
with torch.no_grad():
    test_outputs = best_svm_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...
alpha = 0.000100 | Validation Macro F1: 0.6286

=> Quá trình huấn luyện và tuning hoàn tất trong 1.50 giây.
=> Cấu hình tốt nhất: alpha = 0.0001 với Validation Macro F1: 0.6286

--- Đánh giá Model Multi-class SVM Tốt nhất (alpha=0.0001) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9250    0.8530    0.8875     20520
           1     0.0035    0.0036    0.0035       281
           2     0.8399    0.2691    0.4076      2709
           3     0.5927    0.7521    0.6629      2763
           4     0.5975    0.7500    0.6651      1132
           5     0.4122    0.1669    0.2376      1210
           6     0.5876    0.9592    0.7287      5048
           7     0.9286    0.7738    0.8442       420

    accuracy                         0.7784     34083
   macro avg     0.6109    0.5660    0.5547     34083
weighted avg     0.8047    0.7784

Random Forest

In [32]:
from xgboost import XGBRFClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử trên các max_depth khác nhau
max_depth_values = [15,20,25,30]

best_depth = None
best_f1 = -1
best_rf_model = None

for depth in max_depth_values:
    # Sử dụng XGBRFClassifier với cấu hình device='cuda' để chạy trên GPU
    rf_model = XGBRFClassifier(
        n_estimators=100, 
        max_depth=depth,
        n_jobs=-1, 
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    rf_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính điểm Macro F1
    y_valid_pred = rf_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_rf_model = rf_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Random Forest Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_rf_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...
max_depth = 15 | Validation Macro F1: 0.6256
max_depth = 20 | Validation Macro F1: 0.6287
max_depth = 25 | Validation Macro F1: 0.6295
max_depth = 30 | Validation Macro F1: 0.6316

=> Quá trình huấn luyện và tuning hoàn tất trong 128.79 giây.
=> Cấu hình tốt nhất: max_depth = 30 với Validation Macro F1: 0.6316

--- Đánh giá mô hình Random Forest Tốt nhất (max_depth=30) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9328    0.9935    0.9622     20520
           1     0.0420    0.0178    0.0250       281
           2     0.5025    0.2595    0.3423      2709
           3     0.5335    0.5512    0.5422      2763
           4     0.6101    0.8419    0.7075      1132
           5     0.7565    0.4826    0.5893      1210
           6     0.9483    0.9596    0.9539      5048
           7     0.8596    0.8452    0.8523       420

    accuracy              

Decision Tree

In [30]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử nghiệm các giá trị max_depth khác nhau
max_depth_values = [25]

best_depth = None
best_f1 = -1
best_dt_model = None

for depth in max_depth_values:
    # Dùng XGBClassifier với n_estimators=1 để thiết lập giống một Decision Tree chạy bằng GPU
    dt_model = XGBClassifier(
        n_estimators=1,
        learning_rate=1.0,
        max_depth=depth,
        n_jobs=-1,
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    dt_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính Macro F1
    y_valid_pred = dt_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_dt_model = dt_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_dt_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...
max_depth = 25 | Validation Macro F1: 0.6518

=> Quá trình huấn luyện và tuning hoàn tất trong 0.69 giây.
=> Cấu hình tốt nhất: max_depth = 25 với Validation Macro F1: 0.6518

--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth=25) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9120    0.9931    0.9508     20520
           1     0.0513    0.0285    0.0366       281
           2     0.4385    0.2606    0.3269      2709
           3     0.5579    0.4741    0.5126      2763
           4     0.6511    0.8260    0.7282      1132
           5     0.7616    0.4463    0.5628      1210
           6     0.9346    0.9538    0.9441      5048
           7     0.8440    0.6571    0.7390       420

    accuracy                         0.8499     34083
   macro avg     0.6439    0.5799    0.6001     34083
weighted avg     0.8271    0.8499    0.8334     34083



Logistic Regression

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...")
start_time = time.time()

# Thử nghiệm các alpha và penalty l1, l2
alpha_values = [1e-6]
penalty_values = ['l1']

best_alpha = None
best_penalty = None
best_f1 = -1
best_logreg_model = None

# Khối 1: Chuyển đổi toàn bộ mảng dữ liệu về Numpy (nếu đang ở pandas DataFrame)
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đưa thẳng lên VRAM của GPU với PyTorch
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Định nghĩa mạng: 1 tầng Linear duy nhất + Hàm phạt Cross Entropy = Logistic Regression
class TorchLogisticRegression(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes)
        
    def forward(self, x):
        return self.linear(x)

for penalty in penalty_values:
    for alpha in alpha_values:
        model = TorchLogisticRegression(input_dim, num_classes).to(device)
        
        # Nếu l2 thì dùng tham số weight_decay tích hợp sẵn của Optimizer tự tính toán
        weight_decay = alpha if penalty == 'l2' else 0.0
        optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        epochs = 100
        batch_size = 8192
        num_samples = X_train_t.shape[0]
        
        for epoch in range(epochs):
            # Xáo trộn index bộ nhớ trực tiếp trên GPU để đảm bảo tốc độ tối đa
            indices = torch.randperm(num_samples, device=device)
            
            for i in range(0, num_samples, batch_size):
                idx = indices[i:i+batch_size]
                batch_X = X_train_t[idx]
                batch_y = y_train_t[idx]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                # Nếu l1 thì cộng tay penalty L1 Norm
                if penalty == 'l1':
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + alpha * l1_norm
                    
                loss.backward()
                optimizer.step()
        
        model.eval()
        with torch.no_grad():
            valid_outputs = model(X_valid_t)
            y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
            
        current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
        print(f"penalty = {penalty:2s}, alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
        
        # Chọn model tốt nhất dựa trên Macro F1
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_alpha = alpha
            best_penalty = penalty
            best_logreg_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: penalty = {best_penalty}, alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print(f"\n--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty={best_penalty}, alpha={best_alpha}) trên tập TEST ---")
best_logreg_model.eval()
with torch.no_grad():
    test_outputs = best_logreg_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...
penalty = l1, alpha = 0.000001 | Validation Macro F1: 0.5877

=> Quá trình huấn luyện và tuning hoàn tất trong 1.94 giây.
=> Cấu hình tốt nhất: penalty = l1, alpha = 1e-06 với Validation Macro F1: 0.5877

--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty=l1, alpha=1e-06) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9289    0.9483    0.9385     20520
           1     0.0000    0.0000    0.0000       281
           2     0.4599    0.4976    0.4780      2709
           3     0.5078    0.1299    0.2069      2763
           4     0.5929    0.7668    0.6687      1132
           5     0.3778    0.3041    0.3370      1210
           6     0.7238    0.9584    0.8248      5048
           7     0.9321    0.7190    0.8118       420

    accuracy                         0.8081     34083
   macro avg     0.5654    0.5405 

XGBoost

In [28]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp XGBoost (sử dụng GPU) với bộ tham số đã chọn...")
start_time = time.time()

xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,        # Tăng tốc độ học lên một chút để hội tụ tốt hơn
    max_depth=8,               # Giảm từ 10 xuống 8 để tránh chẻ quá sâu overfit vào một số lớp
    min_child_weight=2,        # Tăng lên 2 để yêu cầu node có đủ mẫu mới chia, giúp tăng Precision
    gamma=0.2,                 # Giảm chi phí phân nhánh
    reg_lambda=1.5,
    reg_alpha=0.5,
    subsample=0.8,         
    colsample_bytree=0.8,
    max_delta_step=2,
    random_state=42,
    eval_metric='mlogloss',
    tree_method="hist",
    device="cuda",
    early_stopping_rounds=100  # Chờ kiên nhẫn hơn
)

# Huấn luyện mô hình, sử dụng tập validation cho early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=100 
)

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện hoàn tất trong {train_time:.2f} giây.")

# Dự đoán và đánh giá trên tập validation
y_valid_pred = xgb_model.predict(X_valid)
valid_f1 = f1_score(y_valid, y_valid_pred, average='macro')
print(f"=> Validation Macro F1: {valid_f1:.4f}")

# Đánh giá trên tập test
print(f"\n--- Đánh giá mô hình XGBoost trên tập TEST ---")
y_test_pred = xgb_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp XGBoost (sử dụng GPU) với bộ tham số đã chọn...
[0]	validation_0-mlogloss:1.26361	validation_1-mlogloss:1.27332
[100]	validation_0-mlogloss:0.27268	validation_1-mlogloss:0.46767
[200]	validation_0-mlogloss:0.25520	validation_1-mlogloss:0.49118
[211]	validation_0-mlogloss:0.25430	validation_1-mlogloss:0.49372

=> Quá trình huấn luyện hoàn tất trong 5.20 giây.
=> Validation Macro F1: 0.6938

--- Đánh giá mô hình XGBoost trên tập TEST ---
              precision    recall  f1-score   support

           0     0.9376    0.9996    0.9676     20520
           1     0.0714    0.0036    0.0068       281
           2     0.5171    0.3562    0.4219      2709
           3     0.5486    0.5820    0.5648      2763
           4     0.6874    0.8410    0.7565      1132
           5     0.7302    0.2661    0.3901      1210
           6     0.9303    0.9628    0.9463      5048
           7     0.9739    0.8000    0.8784       420

    accuracy                         0.8672    